<a href="https://colab.research.google.com/github/mlbyhoang/google-analytics-bigquery-api/blob/main/notebooks/ga4_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Libraries
import pandas as pd
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Filter,
    FilterExpression,
    Metric,
    RunReportRequest,
)

In [ ]:
def get_ga4_report(
    property_id: str,
    page_type: str,
    start_date: str = "30daysAgo",
    end_date: str = "yesterday",
) -> pd.DataFrame:
    """Fetch a paginated GA4 report as a DataFrame."""

    # Initialize API client
    client = BetaAnalyticsDataClient()

    # Define report dimensions and metrics
    dimensions = ["date", "hour", "customEvent:page_type"]
    metrics = ["eventCount", "averageSessionDuration"]

    all_rows = []
    offset = 0
    limit = 100000

    while True:
        # Build API request
        request = RunReportRequest(
            property=f"properties/{property_id}",
            date_ranges=[
                DateRange(
                    start_date=start_date,
                    end_date=end_date,
                )
            ],
            dimensions=[
                Dimension(name=name) for name in dimensions
            ],
            metrics=[
                Metric(name=name) for name in metrics
            ],
            dimension_filter=FilterExpression(
                filter=Filter(
                    field_name="customEvent:page_type",
                    string_filter=Filter.StringFilter(
                        match_type=Filter.StringFilter.MatchType.EXACT,
                        value=page_type,
                    ),
                )
            ),
            limit=limit,
            offset=offset,
        )

        response = client.run_report(request)

        # Collect response data
        for row in response.rows:
            all_rows.append(
                [v.value for v in row.dimension_values]
                + [v.value for v in row.metric_values]
            )

        # Handle pagination
        offset += len(response.rows)

        if not response.rows or offset >= response.row_count:
            break

    # Convert response to DataFrame
    df = pd.DataFrame(
        all_rows,
        columns=dimensions + metrics,
    )

    # Convert metric types
    if not df.empty:
        df["eventCount"] = df["eventCount"].astype(int)
        df["averageSessionDuration"] = df[
            "averageSessionDuration"
        ].astype(float)

    return df


if __name__ == "__main__":
    # Load configuration
    property_id = os.environ["GA4_PROPERTY_ID"]
    page_type = os.getenv("GA4_PAGE_TYPE", "product_detail")

    report = get_ga4_report(
        property_id=property_id,
        page_type=page_type,
    )

    print(f"Retrieved {len(report)} rows.")
    print(report.head())
